# Gold analítico

Construye las tres tablas Gold exclusivamente desde Silver validado. La ejecución es determinista para el cutoff recibido y registra la calidad del cruce Twitch-Steam en Audit.

In [0]:
import json
import uuid

dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("audit_schema", "audit")
dbutils.widgets.text("run_id", "")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
audit_schema = dbutils.widgets.get("audit_schema")
run_id = dbutils.widgets.get("run_id") or str(uuid.uuid4())

In [0]:
%sql
-- El schema se crea de forma idempotente antes de escribir Gold.
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :gold_schema);
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.' || :gold_schema || '.game_growth') (
  app_id BIGINT, month_start DATE, name STRING, release_date DATE,
  avg_players DECIMAL(18,2), peak_players BIGINT, peak_to_avg_ratio DOUBLE,
  prev_avg_players DECIMAL(18,2), mom_change_pct DOUBLE,
  avg_players_ma3 DECIMAL(18,2), months_since_release INT, in_catalog BOOLEAN
) USING DELTA;
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.' || :gold_schema || '.twitch_steam_relationship') (
  app_id BIGINT, month_start DATE, twitch_rank INT, hours_watched BIGINT,
  avg_viewers BIGINT, peak_viewers BIGINT, avg_players_t DECIMAL(18,2),
  avg_players_t1 DECIMAL(18,2), growth_pct_t_to_t1 DOUBLE, match_type STRING
) USING DELTA;
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_game') (
  app_id BIGINT, name STRING, release_date DATE, is_unreleased BOOLEAN,
  price DECIMAL(18,2), owners_min DECIMAL(38,0), owners_max DECIMAL(38,0),
  positive BIGINT, negative BIGINT, review_ratio DOUBLE,
  genres ARRAY<STRING>, developers STRING, publishers STRING
) USING DELTA;

In [0]:
%sql
-- game_growth: evolución mensual por juego.
INSERT OVERWRITE TABLE IDENTIFIER(:catalog || '.' || :gold_schema || '.game_growth')
WITH base AS (
  SELECT
    p.app_id,
    p.month_start,
    p.name,
    p.avg_players,
    p.peak_players,
    p.in_catalog,
    year(p.month_start) * 12 + month(p.month_start) AS month_idx,
    lag(p.month_start) OVER (
      PARTITION BY p.app_id ORDER BY p.month_start
    ) AS prev_month_start,
    lag(p.avg_players) OVER (
      PARTITION BY p.app_id ORDER BY p.month_start
    ) AS prev_avg_players
  FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_player_counts_monthly') p
), moving_average AS (
  SELECT
    b.*,
    avg(b.avg_players) OVER (
      PARTITION BY b.app_id ORDER BY b.month_idx
      RANGE BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS avg_players_ma3_raw,
    count(*) OVER (
      PARTITION BY b.app_id ORDER BY b.month_idx
      RANGE BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS avg_players_ma3_n
  FROM base b
)
SELECT
  m.app_id,
  m.month_start,
  m.name,
  g.release_date,
  m.avg_players,
  m.peak_players,
  try_divide(m.peak_players, m.avg_players) AS peak_to_avg_ratio,
  m.prev_avg_players,
  CASE
    WHEN add_months(m.prev_month_start, 1) = m.month_start
    THEN try_divide(m.avg_players - m.prev_avg_players, m.prev_avg_players)
  END AS mom_change_pct,
  CASE WHEN m.avg_players_ma3_n = 3 THEN m.avg_players_ma3_raw END AS avg_players_ma3,
  CASE
    WHEN g.release_date IS NOT NULL
    THEN try_cast(months_between(m.month_start, trunc(g.release_date, 'MM')) AS INT)
  END AS months_since_release,
  m.in_catalog
FROM moving_average m
LEFT JOIN IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_games') g
  ON m.app_id = g.app_id;

In [0]:
%sql
-- dim_game: dimensión vigente derivada únicamente de silver.games.
INSERT OVERWRITE TABLE IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_game')
SELECT
  app_id,
  name,
  release_date,
  CASE WHEN release_date > current_date() THEN true ELSE false END AS is_unreleased,
  price,
  owners_min,
  owners_max,
  positive,
  negative,
  try_divide(positive, positive + negative) AS review_ratio,
  genres,
  developers,
  publishers
FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_games');

In [0]:
%sql
-- twitch_steam_relationship: solo matches únicos por nombre normalizado.
INSERT OVERWRITE TABLE IDENTIFIER(:catalog || '.' || :gold_schema || '.twitch_steam_relationship')
WITH names AS (
  SELECT name_normalized AS game_name_norm, app_id
  FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_games')
  WHERE name_normalized IS NOT NULL AND trim(name_normalized) <> ''
  UNION
  SELECT lower(trim(name)) AS game_name_norm, app_id
  FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_player_counts_monthly')
  WHERE name IS NOT NULL AND trim(name) <> ''
), name_quality AS (
  SELECT game_name_norm, count(DISTINCT app_id) AS app_count
  FROM names
  GROUP BY game_name_norm
), unique_names AS (
  SELECT n.game_name_norm, min(n.app_id) AS app_id
  FROM names n
  JOIN name_quality q ON n.game_name_norm = q.game_name_norm
  WHERE q.app_count = 1
  GROUP BY n.game_name_norm
), twitch_matches AS (
  SELECT
    t.month_start,
    t.twitch_rank,
    t.game,
    t.game_normalized,
    t.hours_watched,
    t.avg_viewers,
    t.peak_viewers,
    u.app_id,
    CASE
      WHEN u.app_id IS NOT NULL THEN 'unique'
      WHEN q.app_count > 1 THEN 'ambiguous'
      ELSE 'none'
    END AS match_type
  FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.twitch_game_activity_monthly') t
  LEFT JOIN unique_names u ON t.game_normalized = u.game_name_norm
  LEFT JOIN name_quality q ON t.game_normalized = q.game_name_norm
), unique_matches AS (
  SELECT
    x.*,
    row_number() OVER (
      PARTITION BY x.app_id, x.month_start
      ORDER BY x.twitch_rank, x.game_normalized
    ) AS rn
  FROM twitch_matches x
  WHERE x.match_type = 'unique'
), with_players AS (
  SELECT
    u.app_id,
    u.month_start,
    u.twitch_rank,
    u.hours_watched,
    u.avg_viewers,
    u.peak_viewers,
    p0.avg_players AS avg_players_t,
    p1.avg_players AS avg_players_t1,
    try_divide(p1.avg_players - p0.avg_players, p0.avg_players) AS growth_pct_t_to_t1,
    u.match_type
  FROM unique_matches u
  LEFT JOIN IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_player_counts_monthly') p0
    ON u.app_id = p0.app_id AND u.month_start = p0.month_start
  LEFT JOIN IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_player_counts_monthly') p1
    ON u.app_id = p1.app_id AND add_months(u.month_start, 1) = p1.month_start
  WHERE u.rn = 1
)
SELECT * FROM with_players;

In [0]:
%sql
-- Audit: cobertura del matching Twitch-Steam por filas y horas vistas.
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.' || :audit_schema || '.match_quality') (
  run_id STRING, match_type STRING, twitch_rows BIGINT,
  distinct_twitch_games BIGINT, hours_watched BIGINT, generated_at TIMESTAMP
) USING DELTA;
INSERT OVERWRITE TABLE IDENTIFIER(:catalog || '.' || :audit_schema || '.match_quality')
WITH names AS (
  SELECT name_normalized AS game_name_norm, app_id
  FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_games')
  WHERE name_normalized IS NOT NULL AND trim(name_normalized) <> ''
  UNION
  SELECT lower(trim(name)) AS game_name_norm, app_id
  FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_player_counts_monthly')
  WHERE name IS NOT NULL AND trim(name) <> ''
), quality AS (
  SELECT game_name_norm, count(DISTINCT app_id) AS app_count
  FROM names
  GROUP BY game_name_norm
), classified AS (
  SELECT
    CASE WHEN q.app_count = 1 THEN 'unique'
         WHEN q.app_count > 1 THEN 'ambiguous'
         ELSE 'none' END AS match_type,
    t.hours_watched,
    t.game
  FROM IDENTIFIER(:catalog || '.' || :silver_schema || '.twitch_game_activity_monthly') t
  LEFT JOIN quality q ON t.game_normalized = q.game_name_norm
)
SELECT
  :run_id AS run_id,
  match_type,
  count(*) AS twitch_rows,
  count(DISTINCT game) AS distinct_twitch_games,
  coalesce(sum(hours_watched), 0) AS hours_watched,
  current_timestamp() AS generated_at
FROM classified
GROUP BY match_type;

In [0]:
dbutils.notebook.exit(json.dumps({"status": "completed", "tables": ["game_growth", "twitch_steam_relationship", "dim_game"], "run_id": run_id}))